# Elo Monte Carlo playground
This notebook runs entirely offline. It models a configurable number of head-to-head matches; match labels and tournament stages are intentionally ignored.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

SEED = 42
PLAYERS = 15
MATCHES_PER_TOURNAMENT = 37  # set this to the total number of matches you expect to play
TRUE_SKILL_SPREAD = 180
RUNS = 5_000


In [ ]:
def expected(a, b, scale):
    return 1 / (1 + 10 ** ((b - a) / scale))

def elo_change(winner, loser, margin, target, k, scale, margin_weight):
    return round(k * (1 - expected(winner, loser, scale)) * (1 + margin_weight * margin / target))

def play_match(rng, truth, elo, a, b, target, k, scale, margin_weight):
    a_wins = rng.random() < expected(truth[a], truth[b], scale)
    winner, loser = (a, b) if a_wins else (b, a)
    probability = expected(abs(truth[a] - truth[b]), 0, scale)
    margin = int(rng.integers(2, min(target, 2 + round(probability * target)) + 1))
    change = elo_change(elo[winner], elo[loser], margin, target, k, scale, margin_weight)
    elo[winner] += change
    elo[loser] -= change

def simulate(k=32, scale=400, margin_weight=1):
    rng = np.random.default_rng(SEED)
    final_ratings, correlations = [], []
    for _ in range(RUNS):
        truth = rng.normal(1000, TRUE_SKILL_SPREAD, PLAYERS)
        elo = np.full(PLAYERS, 1000.0)
        for _ in range(MATCHES_PER_TOURNAMENT):
            a, b = rng.choice(PLAYERS, 2, replace=False)
            play_match(rng, truth, elo, a, b, 11, k, scale, margin_weight)
        final_ratings.extend(elo)
        correlations.append(np.corrcoef(truth, elo)[0, 1])
    return np.array(final_ratings), np.mean(correlations)


In [ ]:
candidates = []
for k, scale, margin_weight in product((16, 24, 32, 40), (300, 400, 500), (0, 0.5, 1)):
    ratings, correlation = simulate(k, scale, margin_weight)
    candidates.append((correlation, k, scale, margin_weight, ratings.std()))

for row in sorted(candidates, reverse=True)[:8]:
    print(f'correlation={row[0]:.3f}  K={row[1]} scale={row[2]} margin={row[3]} spread={row[4]:.1f}')

ratings, _ = simulate()
plt.hist(ratings, bins=45, color='#007e6a')
plt.title('Possible player Elo after this tournament format')
plt.xlabel('Elo'); plt.ylabel('Simulated player outcomes')
plt.show()


Set MATCHES_PER_TOURNAMENT to the total number of matches you expect to play, then use the printed candidates as a starting point. Each simulated match selects two distinct players uniformly at random, so this models rating behavior rather than a real fixture, Swiss, or bracket assignment. The chosen values still belong in the website settings; this notebook never writes tournament data.
